# S2F Model Evaluation

This notebook shows how to evaluate a trained Shape2Force (S2F) model on your dataset.

**Metrics computed:**
- **MSE** – Mean Squared Error
- **MS-SSIM** – Multi-Scale Structural Similarity
- **Pixel Correlation** – Pearson correlation between predicted and ground-truth heatmaps
- **Relative Magnitude Error** – WFM-style weighted relative error
- **Force Sum/Mean Correlation** – Correlation of total force per sample

Run the cells below after adjusting paths and settings.

In [ ]:
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')
import sys
import os
import cv2
cv2.utils.logging.setLogLevel(cv2.utils.logging.LOG_LEVEL_ERROR)

cwd = os.getcwd()
S2F_ROOT = cwd if os.path.exists(os.path.join(cwd, 'models')) else os.path.dirname(cwd)
sys.path.insert(0, S2F_ROOT)

import torch
import matplotlib.pyplot as plt

from data.cell_dataset import prepare_data, load_folder_data
from models.s2f_model import create_s2f_model, compute_settings_normalization
from utils.metrics import (
    evaluate_metrics_on_dataset,
    print_metrics_report,
    gen_prediction_plots,
    plot_predictions,
)

print(f"S2F root: {S2F_ROOT}")

## Configuration

In [ ]:
# --- Adjust these paths ---
USE_SINGLE_CELL = True  # True = single-cell (with substrate), False = spheroid
DATA_FOLDER = os.path.join(S2F_ROOT, 'sample')  # or path to your dataset
CHECKPOINT_PATH = os.path.join(S2F_ROOT, 'ckp', 'best_checkpoint.pth')

IMAGE_SIZE = 1024
BATCH_SIZE = 2
THRESHOLD = 0.0  # Threshold for heatmap metrics (0 = no threshold)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SUBSTRATE = 'fibroblasts_PDMS'  # For single-cell mode

## Load Data

In [ ]:
# Option A: Load from folder with train/test subfolders
train_folder = os.path.join(DATA_FOLDER, 'train')
test_folder = os.path.join(DATA_FOLDER, 'test')

if os.path.exists(train_folder) and os.path.exists(test_folder):
    train_loader, val_loader = prepare_data(
        DATA_FOLDER,
        batch_size=BATCH_SIZE,
        target_size=(IMAGE_SIZE, IMAGE_SIZE),
        use_augmentations=False,
        train_test_sep_folder=True,
        return_metadata=USE_SINGLE_CELL,
        substrate=SUBSTRATE if USE_SINGLE_CELL else None,
    )
    print(f"Loaded train: {len(train_loader.dataset)} samples, val: {len(val_loader.dataset)} samples")
else:
    # Option B: Load from a single folder (e.g. test only)
    val_loader = load_folder_data(
        DATA_FOLDER,
        substrate=SUBSTRATE if USE_SINGLE_CELL else None,
        img_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        return_metadata=USE_SINGLE_CELL,
    )
    train_loader = None
    print(f"Loaded {len(val_loader.dataset)} samples from {DATA_FOLDER}")

## Load Model

In [ ]:
in_channels = 3 if USE_SINGLE_CELL else 1
generator, _ = create_s2f_model(in_channels=in_channels)
checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
generator.load_state_dict(checkpoint.get('generator_state_dict', checkpoint), strict=True)
generator = generator.to(DEVICE)
generator.eval()

print(f"Loaded checkpoint from {CHECKPOINT_PATH}")
if 'epoch' in checkpoint:
    print(f"  Epoch: {checkpoint['epoch']}")

## Run Evaluation

In [ ]:
config_path = os.path.join(S2F_ROOT, 'config', 'substrate_settings.json')
normalization_params = compute_settings_normalization(config_path=config_path) if USE_SINGLE_CELL else None

val_results = evaluate_metrics_on_dataset(
    generator,
    val_loader,
    device=DEVICE,
    description="Evaluating",
    save_predictions=True,
    threshold=THRESHOLD,
    use_settings=USE_SINGLE_CELL,
    normalization_params=normalization_params,
    config_path=config_path,
    substrate_override=SUBSTRATE,
)

report = {'validation': val_results}
if train_loader is not None:
    train_results = evaluate_metrics_on_dataset(
        generator, train_loader, device=DEVICE, description="Training",
        threshold=THRESHOLD, use_settings=USE_SINGLE_CELL,
        normalization_params=normalization_params, config_path=config_path,
        substrate_override=SUBSTRATE,
    )
    report = {'train': train_results, 'validation': val_results}

print_metrics_report(report, threshold=THRESHOLD)

## Visualize Predictions

In [ ]:
# Quick preview: plot first few samples
plot_predictions(
    val_loader,
    generator,
    n_samples=3,
    device=DEVICE,
    use_settings=USE_SINGLE_CELL,
    normalization_params=normalization_params,
    config_path=config_path,
    substrate_override=SUBSTRATE,
)

In [ ]:
# Save individual prediction plots (sorted by MSE, best first)
plot_output_dir = os.path.join(S2F_ROOT, 'outputs', 'evaluation_plots')
if 'individual_predictions' in val_results:
    gen_prediction_plots(
        val_results['individual_predictions'],
        save_dir=plot_output_dir,
        sort_by='mse',
        sort_order='asc',
        threshold=THRESHOLD,
    )
    print(f"Saved plots to {plot_output_dir}")